# Member 4: spectrum_formula.arch2_contrastive, ensemble + PubChem channel (inference only, offline)

Averages the scaled cosines of four saved arch2 models, applies the library-search gate, then adds a PubChem candidate
channel for molecules the gate leaves to the model: mass window over a mass-sorted PubChem catalogue, ensemble score,
(PubChem channel on) and `submission_gate_only.csv` (without it); `submission.csv` applies PubChem to every molecule (no gate).
(PubChem channel on) and `submission_gate_only.csv` (without it).

**Attach before running:** the competition data and your private datasets `casmi-m4-code`, `casmi26-coconut`,
`rdkit-wheel`, `casmi-m4-weights`, `casmi26-pubchem-index`. Internet off; GPU on.


In [ ]:
import glob
import importlib
import os
import shutil
import subprocess
import sys
import time

import pandas as pd

T0 = time.time()
QUICK = os.environ.get("M4_QUICK", "0") == "1"      # 1 = tiny wiring check

def find(pattern):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"nothing matching {pattern} under /kaggle/input; attach the dataset")
    return hits[0]

CODE = os.path.dirname(find("CASMI_M4_CODE"))
WEIGHTS = os.path.dirname(find("CASMI_M4_WEIGHTS"))      # trained checkpoints (private dataset)
CAND_DIR = "/kaggle/working/candidates"   # candidate tables from separate datasets, gathered in one folder
os.makedirs(CAND_DIR, exist_ok=True)
for name in ("coconut.parquet", "pubchemlite.parquet", "pubchemlite_counts.parquet"):
    try:
        shutil.copy(find(name), CAND_DIR)
    except FileNotFoundError:
        print(f"{name} not attached")
PUBCHEM_INDEX = find("pubchem_index.parquet")   # mass-sorted PubChem catalogue (private dataset)
sys.path[:0] = [f"{CODE}/src", f"{CODE}/models/spectrum_formula/arch2_contrastive"]
os.environ.update(CASMI_ENV="kaggle", CASMI_CANDIDATES_DIR=CAND_DIR)
print("code:", CODE, "| candidates:", CAND_DIR, "| python", sys.version.split()[0])

In [ ]:
# RDKit is not in Kaggle's image: install the wheel that matches this Python, offline.
try:
    import rdkit  # noqa: F401
except ImportError:
    tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
    wheel = find(f"rdkit*{tag}*.whl")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--no-deps", wheel], check=True)
    importlib.invalidate_caches()
import rdkit

print("RDKit", rdkit.__version__)   # the metric pins 2026.03.3

In [ ]:
import torch

NAMES = ["c1", "c2", "e20", "pcl"]                       # saved models to average (any subset of these four)
ckpts = [f"{WEIGHTS}/arch2_{n}.pt" for n in NAMES]
cfg = torch.load(ckpts[0], map_location="cpu", weights_only=False)["cfg"]   # settings saved with model 1
cfg["data"].update(data_dir=None, n_val=3000, external_pools=["coconut"])
EXTRA = {}         # per-run data overrides, e.g. {"external_pools": ["coconut", "pubchemlite"]}
RUN_GATE = True    # fuse library search (B2) where it is reliable (Member 7's gate)
cfg["data"].update(EXTRA)
cfg["train"]["device"] = "auto"
cfg["predict"]["ensemble"] = {"checkpoints": ckpts}
cfg["predict"]["prior"] = None
cfg["predict"]["retrieval"] = None   # an ensemble already captures what retrieval adds (+0.001 on top)
EXTRA_PREDICT = {}  # per-run predict overrides, e.g. {"retrieval": {"weight": 20.0, "k": 20, "beta": 30.0}}
cfg["predict"].update(EXTRA_PREDICT)
print(NAMES, "| GPUs:", torch.cuda.device_count(), "| pool:", cfg["data"]["external_pools"])

In [ ]:
# Team protocol: split, pool (+ COCONUT) and pool fingerprints are built once and reused below.
import train as m4_train

proto = m4_train.build_protocol(cfg)
n_test = proto.test.molecule_id.nunique()
print(f"pool {len(proto.pool):,} structures | val molecules {len(proto.truth):,} | test molecules {n_test:,}")

In [ ]:
# Validation on the team protocol, then the test submission (optionally gated with library search).
import predict as m4_predict
from common import b2gate

cp = proto.cp
pool_keys = proto.pool.key.to_numpy()
val_rank, _, truth, _ = m4_predict.predict(cfg, "val", proto)
test_rank, _, _, _ = m4_predict.predict(cfg, "test", proto)
tables = {"model": cp.evaluate(val_rank, truth, pool_keys)}
honest = {"model": float(tables["model"].loc["balanced", "MRR@25"])}
final_rank, chosen, gate_info = test_rank, "model", None
print("model only:\n", tables["model"].to_string())

if RUN_GATE:
    val_g, test_g, gate_info, _ = b2gate.fuse_with_b2(proto, val_rank, test_rank, log=print)
    tables["gated"] = cp.evaluate(val_g, truth, pool_keys)
    honest["gated"] = gate_info["cross_fitted"]          # tau was tuned on validation: judge it out of sample
    print("gate:", gate_info, "\ngated:\n", tables["gated"].to_string())
    if honest["gated"] > honest["model"]:
        final_rank, chosen = test_g, "gated"

cp.write_submission(test_rank, proto.pool, proto.test, path="/kaggle/working/submission_model_only.csv")
sub = cp.write_submission(final_rank, proto.pool, proto.test, path="/kaggle/working/submission.csv")
pd.concat(tables, names=["method"]).to_csv("/kaggle/working/validation_table.csv")
print(f"submission uses: {chosen} | honest val {honest} | {sub.shape} | total {(time.time() - T0) / 60:.1f} min")
sub.head()

## PubChem channel
Flat prior chosen on a pool-removal validation (answers absent from the pool must be recovered from PubChem).
Molecules where the library gate fired keep the gated list (library evidence is strong there).


In [ ]:
import numpy as np
from common import contrastive as C
from common import features as F
from common import pubchem_funnel as PF
from common import team

from casmi.candidates.pool import load_pool
from casmi.candidates.pubchem_index import MassIndex

USE_PUBCHEM = True
PC_WEIGHT, PC_COCONUT, PC_LIBRARY = 1.0, 4.0, 0.0   # ln(1+PubMed) weight, COCONUT bonus, library bonus (flat prior)
device = m4_predict.pick_device("auto")
models = C.load_checkpoints(ckpts, cp.FP_BITS, device, cfg["data"])
arrays = team.spectra_arrays(proto.test_path, proto.test, **F.spec_args(cfg["data"]))
coconut_keys = set(load_pool("coconut.parquet").inchikey14.astype(str))
union_rank, ext_pool = PF.union_rankings(
    proto, proto.test, arrays, models, device, MassIndex(PUBCHEM_INDEX), coconut_keys,
    weight=PC_WEIGHT, coconut_bonus=PC_COCONUT, library_bonus=PC_LIBRARY, chunk=50)
n_pool = len(proto.pool)
gated_mols = {m for m in test_rank if final_rank[m] != test_rank[m]} if RUN_GATE else set()
pc_final = {m: (final_rank[m] if m in gated_mols else union_rank[m]) for m in test_rank}
n_union = len(pc_final) - len(gated_mols)
print(f"{len(gated_mols)} molecules keep the gated list; {n_union} use the PubChem union ranking; "
      f"{len(ext_pool) - n_pool:,} new PubChem structures reached a top 25 | {(time.time() - T0) / 60:.1f} min")

In [ ]:
# Diagnostics (no spectra-independent tricks: how much of the output is PubChem-only?) and the files to inspect
keys = ext_pool.key.to_numpy()
rep = pd.DataFrame({"molecule_id": list(pc_final), "gated": [m in gated_mols for m in pc_final],
                    "pubchem_only_in_top25": [sum(i >= n_pool for i in pc_final[m][:25]) for m in pc_final],
                    "top1_pubchem_only": [pc_final[m][0] >= n_pool for m in pc_final]})
rep.to_csv("/kaggle/working/pubchem_report.csv", index=False)
top25 = [";".join(keys[i] for i in union_rank[m][:25]) for m in union_rank]
pd.DataFrame({"molecule_id": list(union_rank), "top25_keys": top25}
             ).to_csv("/kaggle/working/union_top25_keys.csv", index=False)   # compared with the local dry run
print(rep.describe().T[["mean", "min", "max"]].to_string())
cp.write_submission(pc_final, ext_pool, proto.test, path="/kaggle/working/submission_gate_plus_pubchem.csv")
cp.write_submission(final_rank, proto.pool, proto.test, path="/kaggle/working/submission_gate_only.csv")
# PubChem union ranking for EVERY molecule (no library gate): the clean test of the channel
cp.write_submission(union_rank, ext_pool, proto.test, path="/kaggle/working/submission.csv")
print(f"done in {(time.time() - T0) / 60:.1f} min")